# Audio Recording & Acoustic Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/main/tutorials/audio/audio_recording_and_acoustic_analysis.ipynb)

In this tutorial you will learn how to:

1. **Record or load audio** into senselab's `Audio` data structure
2. **Visualize waveforms and spectrograms** to see what sound "looks like"
3. **Extract pitch (F0)** to measure how high or low a voice sounds
4. **Extract formants (F1, F2)** to characterize vowel quality and vocal tract shape
5. **Detect emotion** from speech using a pre-trained model
6. **Compare two speakers** to determine whether audio samples come from the same person

7. **Enhance speech** and compare acoustic features before and after enhancement

These are foundational skills for voice and speech analysis in behavioral research.

In [ ]:
# Install senselab and dependencies
!pip install -q uv
!uv pip install --pre --system "senselab"

> **\u26a0\ufe0f Restart runtime after install**
>
> The install may upgrade packages (e.g., numpy) that are already loaded.
> Go to **Runtime \u2192 Restart session**, then **run all cells below** (skip this install cell).

In [ ]:
import os

import matplotlib.pyplot as plt
import torch
import torchaudio.transforms as T
from senselab.audio.data_structures import Audio
from senselab.audio.tasks.classification.speech_emotion_recognition.api import (
    classify_emotions_from_speech,
)
from senselab.audio.tasks.features_extraction.praat_parselmouth import (
    extract_praat_parselmouth_features_from_audios,
)
from senselab.audio.tasks.features_extraction.torchaudio import (
    extract_pitch_from_audios,
)
from senselab.audio.tasks.plotting.plotting import (
    play_audio,
    plot_specgram,
    plot_waveform,
)
from senselab.audio.tasks.preprocessing.preprocessing import (
    downmix_audios_to_mono,
    resample_audios,
)
from senselab.audio.tasks.speaker_verification.speaker_verification import (
    verify_speaker,
)
from senselab.utils.data_structures import DeviceType, HFModel, SpeechBrainModel

# Auto-detect device
device = DeviceType.CUDA if torch.cuda.is_available() else DeviceType.CPU
print(f"Using device: {device.value}")

## 1. Record or Load Audio

Audio is a digital representation of sound waves. Key concepts:

- **Waveform**: a sequence of amplitude values sampled at regular intervals. Each value
  represents the air pressure displacement at that instant.
- **Sampling rate**: how many samples are captured per second. For example, 16 000 Hz
  means 16 000 amplitude values per second of audio.
- **Channels**: mono (1 channel) captures a single signal; stereo (2 channels) captures
  left and right signals separately.

Below you can either **record your own voice** (in Google Colab) or **download
a sample file**. The recording widget uses your browser's microphone; if you're
not in Colab or prefer not to record, a sample file is loaded automatically.

In [ ]:
# Option 1: Record audio in Google Colab (uses browser microphone)
# Option 2: Automatically falls back to downloading a sample file
import urllib.request
from pathlib import Path

Path("tutorial_audio_files").mkdir(exist_ok=True)

base_url = "https://github.com/sensein/senselab/raw/main/src/tests/data_for_testing"
audio_path = None

try:
    from google.colab import output
    from IPython.display import HTML, display

    # JavaScript-based recording widget for Colab
    RECORD_JS = """
    <button id="record-btn" style="font-size:16px;padding:8px 16px">🎙 Click to Record (3s)</button>
    <script>
    (async () => {
        const btn = document.getElementById('record-btn');
        btn.onclick = async () => {
            btn.textContent = '🔴 Recording...';
            const stream = await navigator.mediaDevices.getUserMedia({audio: true});
            const recorder = new MediaRecorder(stream);
            const chunks = [];
            recorder.ondataavailable = e => chunks.push(e.data);
            recorder.onstop = async () => {
                stream.getTracks().forEach(t => t.stop());
                const blob = new Blob(chunks, {type: 'audio/wav'});
                const reader = new FileReader();
                reader.onload = () => {
                    const b64 = reader.result.split(',')[1];
                    google.colab.kernel.invokeFunction('save_recording', [b64], {});
                };
                reader.readAsDataURL(blob);
                btn.textContent = '✅ Recorded! Saved.';
            };
            recorder.start();
            setTimeout(() => recorder.stop(), 3000);
        };
    })();
    </script>
    """

    def save_recording(b64_data):
        import base64

        raw = base64.b64decode(b64_data)
        with open("tutorial_audio_files/my_recording.wav", "wb") as f:
            f.write(raw)

    output.register_callback("save_recording", save_recording)
    display(HTML(RECORD_JS))
    print("Click the button above to record 3 seconds of audio.")
    print("After recording, re-run this cell to load it.")

    rec_path = Path("tutorial_audio_files/my_recording.wav")
    if rec_path.exists() and rec_path.stat().st_size > 1000:
        audio_path = str(rec_path.resolve())
        print(f"Using your recording: {audio_path}")
except Exception:
    pass

# Fallback: download sample audio files
for fname in ["audio_48khz_mono_16bits.wav", "audio_48khz_stereo_16bits.wav"]:
    dest = Path("tutorial_audio_files") / fname
    if not dest.exists():
        urllib.request.urlretrieve(f"{base_url}/{fname}", str(dest))

if audio_path is None:
    audio_path = os.path.abspath("tutorial_audio_files/audio_48khz_mono_16bits.wav")
    print(f"Using sample audio: {audio_path}")

audio = Audio(filepath=audio_path)
print(f"Loaded audio: {audio.waveform.shape[1]} samples at {audio.sampling_rate} Hz")
print(f"Duration: {audio.waveform.shape[1] / audio.sampling_rate:.2f} seconds")
print(f"Channels: {audio.waveform.shape[0]}")

## 2. Waveform Visualization

A waveform plot shows amplitude (loudness) over time. Positive and negative values
represent the air pressure changes that make up sound. Louder sounds have larger
amplitude swings; quiet passages stay close to zero.

In [ ]:
plot_waveform(audio)
plt.show()

play_audio(audio)

## 3. Spectrogram

A spectrogram shows how the frequency content of audio changes over time. Think of
it as a "heat map" where:

- **X-axis** = time
- **Y-axis** = frequency
- **Color** = intensity (energy at that frequency and time)

The **mel scale** is a perceptual frequency scale that better matches how humans
hear -- lower frequencies are spaced more widely, reflecting the fact that we are
more sensitive to differences at low frequencies than at high ones.

In [ ]:
# Linear frequency spectrogram
plot_specgram(audio, title="Linear Spectrogram")
plt.show()

# Mel-scale spectrogram (better matches human hearing)
plot_specgram(audio, mel_scale=True, title="Mel Spectrogram")
plt.show()

## 4. Pitch (F0) Extraction

**Pitch** (fundamental frequency, F0) is the rate at which your vocal folds vibrate.
It determines how "high" or "low" your voice sounds.

- Typical range: ~80 Hz (deep voice) to ~300 Hz (high voice)
- **Pitch contour** -- how pitch changes over time -- carries information about
  intonation, stress, and emotion.

Below we resample to 16 kHz (a standard rate for many speech models) and extract
the pitch contour using normalized cross-correlation.

In [ ]:
# Resample to 16 kHz for pitch extraction
audio_16k = resample_audios([audio], resample_rate=16000)[0]

# Extract pitch contour
pitch_result = extract_pitch_from_audios([audio_16k], freq_low=80, freq_high=500)
pitch_contour = pitch_result[0]["pitch"].squeeze()

# Build a time axis (default hop length for torchaudio pitch detection)
hop_length = 160  # default for 16 kHz
time_axis = torch.arange(len(pitch_contour)) * hop_length / 16000

# Plot only voiced frames (pitch > 0)
plt.figure(figsize=(12, 4))
voiced = pitch_contour > 0
plt.scatter(time_axis[voiced], pitch_contour[voiced], s=2, c="blue")
plt.xlabel("Time (seconds)")
plt.ylabel("Frequency (Hz)")
plt.title("Pitch (F0) Contour")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Formant Extraction

**Formants** are resonant frequencies of the vocal tract -- the natural
amplification that occurs at certain frequencies due to the shape of your mouth,
throat, and nasal cavities.

- **F1** (first formant): correlates with tongue height. Low F1 = high tongue
  (as in "ee"); high F1 = low tongue (as in "ah").
- **F2** (second formant): correlates with tongue front/back position. High F2 =
  front vowel ("ee"); low F2 = back vowel ("oo").

Together, F1 and F2 can distinguish most vowels -- this is why a vowel space plot
(F1 vs F2) is a standard tool in phonetics.

Below we overlay pitch (F0) and formant tracks (F1, F2) on the spectrogram,
showing how these features relate to the spectral energy distribution.

In [ ]:
# Extract pitch and formants, then overlay on spectrogram
import parselmouth
import numpy as np

snd = parselmouth.Sound(audio.waveform.squeeze().numpy(), sampling_frequency=audio.sampling_rate)
formant = snd.to_formant_burg()

# Get F1 and F2 at each time frame
formant_times = np.array(formant.xs())
f1_values = np.array([formant.get_value_at_time(1, t) for t in formant_times])
f2_values = np.array([formant.get_value_at_time(2, t) for t in formant_times])

# Create spectrogram with pitch and formant overlay
waveform_np = audio.waveform.squeeze().numpy()
duration = len(waveform_np) / audio.sampling_rate

fig, ax = plt.subplots(figsize=(14, 6))

# Spectrogram background
spec_transform = T.MelSpectrogram(sample_rate=audio.sampling_rate, n_fft=2048, hop_length=512, n_mels=128, f_max=5500)
spec = spec_transform(audio.waveform.squeeze())
spec_db = T.AmplitudeToDB()(spec)
ax.imshow(spec_db.numpy(), aspect="auto", origin="lower", extent=[0, duration, 0, 5500], cmap="magma", alpha=0.9)

# Overlay pitch contour
pitch_time = torch.arange(len(pitch_contour)) * hop_length / 16000
voiced = pitch_contour > 0
ax.scatter(pitch_time[voiced].numpy(), pitch_contour[voiced].numpy(), s=3, c="cyan", label="Pitch (F0)", zorder=3)

# Overlay formant tracks
valid_f1 = ~np.isnan(f1_values)
ax.scatter(formant_times[valid_f1], f1_values[valid_f1], s=1, c="lime", alpha=0.5, label="F1", zorder=2)
valid_f2 = ~np.isnan(f2_values)
ax.scatter(formant_times[valid_f2], f2_values[valid_f2], s=1, c="yellow", alpha=0.5, label="F2", zorder=2)

ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("Spectrogram with Pitch and Formant Overlay")
ax.legend(loc="upper right", fontsize=8)
ax.set_ylim(0, 5500)
plt.tight_layout()
plt.show()

# Print summary statistics
print("=== Formant Summary ===")
print(f"  F1 mean: {np.nanmean(f1_values):.0f} Hz (std: {np.nanstd(f1_values):.0f} Hz)")
print(f"  F2 mean: {np.nanmean(f2_values):.0f} Hz (std: {np.nanstd(f2_values):.0f} Hz)")

## 6. Speech Emotion Recognition

Speech emotion recognition (SER) uses machine learning to detect the emotional
state of a speaker from acoustic cues -- not the words themselves, but **how**
they are spoken. Features like pitch variability, speech rate, energy patterns,
and spectral characteristics all carry emotional information.

We use a pre-trained wav2vec2-based model fine-tuned for emotion classification.

In [ ]:
# Classify emotion from speech
model = HFModel(path_or_uri="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition")
results = classify_emotions_from_speech([audio_16k], model=model, device=device)

print("=== Emotion Classification ===")
for label, score in zip(results[0].labels, results[0].scores):
    bar = "\u2588" * int(score * 40)
    print(f"  {label:12s}: {score:.3f} {bar}")

## 7. Speaker Matching

Speaker verification determines whether two audio samples come from the same
person. It works by:

1. Extracting a **speaker embedding** -- a compact numerical representation of
   voice characteristics (timbre, pitch range, speaking style).
2. Comparing embeddings using **cosine similarity**.
3. If similarity exceeds a threshold, the speakers are likely the same person.

Below we demonstrate two cases:
1. **Same speaker**: Split one recording into two halves and verify they match.
2. **Different speakers**: Compare recordings from two different people.

This shows how the similarity score behaves in both scenarios.

In [ ]:
# Split the first recording into two halves (same speaker)
mid = audio_16k.waveform.shape[1] // 2
audio_part1 = Audio(waveform=audio_16k.waveform[:, :mid], sampling_rate=16000)
audio_part2 = Audio(waveform=audio_16k.waveform[:, mid:], sampling_rate=16000)

# Load a different speaker's audio
audio2 = Audio(filepath=os.path.abspath("tutorial_audio_files/audio_48khz_stereo_16bits.wav"))
audio2_mono = downmix_audios_to_mono([audio2])[0]
audio2_16k = resample_audios([audio2_mono], resample_rate=16000)[0]

# Compare: same speaker (two halves of same recording)
same_score, same_match = verify_speaker([(audio_part1, audio_part2)])[0]

# Compare: different speakers
diff_score, diff_match = verify_speaker([(audio_16k, audio2_16k)])[0]

print("=== Speaker Verification ===")
print()
print("Same speaker (two halves of one recording):")
print(f"  Similarity: {same_score:.4f} -> {'MATCH' if same_match else 'NO MATCH'}")
print()
print("Different speakers (two different recordings):")
print(f"  Similarity: {diff_score:.4f} -> {'MATCH' if diff_match else 'NO MATCH'}")
print()
print("(Threshold: 0.25 -- scores above indicate same speaker)")

# Listen to the two audio clips being compared
print("\nAudio 1 (first half):")
play_audio(audio_part1)

In [ ]:
print("Audio 2 (different recording):")
play_audio(audio2_16k)

## 8. Speech Enhancement: Before and After

**Speech enhancement** removes background noise and improves signal quality
using neural network models. This is useful as a preprocessing step before
running analysis on noisy recordings.

Below we enhance the audio and compare the acoustic features (spectrogram,
pitch, formants) before and after enhancement to see how the model affects
the signal.

In [ ]:
# Enhance the audio using SpeechBrain's speech enhancement model
from senselab.audio.tasks.speech_enhancement.api import enhance_audios

enhanced_audio = enhance_audios(
    [audio_16k], model=SpeechBrainModel(path_or_uri="speechbrain/sepformer-wham16k-enhancement"), device=device
)[0]
print(f"Enhanced audio: {enhanced_audio.waveform.shape[1]} samples at {enhanced_audio.sampling_rate} Hz")

# Compare spectrograms: before vs after enhancement
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Before enhancement ---
wf_orig = audio_16k.waveform.squeeze().numpy()
dur_orig = len(wf_orig) / audio_16k.sampling_rate

# Spectrogram (before)
spec_t = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=256, n_mels=80, f_max=5500)
spec_orig = T.AmplitudeToDB()(spec_t(audio_16k.waveform.squeeze()))
axes[0, 0].imshow(spec_orig.numpy(), aspect="auto", origin="lower", extent=[0, dur_orig, 0, 5500], cmap="magma")
axes[0, 0].set_title("Before Enhancement")
axes[0, 0].set_ylabel("Frequency (Hz)")

# Spectrogram (after)
spec_enh = T.AmplitudeToDB()(spec_t(enhanced_audio.waveform.squeeze()))
dur_enh = enhanced_audio.waveform.shape[1] / enhanced_audio.sampling_rate
axes[0, 1].imshow(spec_enh.numpy(), aspect="auto", origin="lower", extent=[0, dur_enh, 0, 5500], cmap="magma")
axes[0, 1].set_title("After Enhancement")

# Pitch comparison
pitch_orig = extract_pitch_from_audios([audio_16k])[0]["pitch"].squeeze()
pitch_enh = extract_pitch_from_audios([enhanced_audio])[0]["pitch"].squeeze()
t_orig = torch.arange(len(pitch_orig)) * 160 / 16000
t_enh = torch.arange(len(pitch_enh)) * 160 / 16000

v_orig = pitch_orig > 0
axes[1, 0].scatter(t_orig[v_orig], pitch_orig[v_orig], s=2, c="blue", label="Before")
axes[1, 0].set_title("Pitch: Before")
axes[1, 0].set_ylabel("F0 (Hz)")
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].grid(True, alpha=0.3)

v_enh = pitch_enh > 0
axes[1, 1].scatter(t_enh[v_enh], pitch_enh[v_enh], s=2, c="green", label="After")
axes[1, 1].set_title("Pitch: After")
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("Speech Enhancement: Before vs After", fontsize=14)
plt.tight_layout()
plt.show()

# Play both for comparison
print("Original audio:")
play_audio(audio_16k)

In [ ]:
print("Enhanced audio:")
play_audio(enhanced_audio)

## Summary

In this tutorial we covered:

- **Loading audio** into senselab's `Audio` object and inspecting its properties
  (sampling rate, duration, channels).
- **Waveform visualization** to see amplitude over time, and **spectrograms**
  (linear and mel-scale) to see frequency content over time.
- **Pitch (F0) extraction** to measure vocal fold vibration rate and plot the
  intonation contour.
- **Formant extraction** (F1, F2) using Praat/Parselmouth to characterize vowel
  quality and vocal tract resonance.
- **Speech emotion recognition** using a pre-trained wav2vec2 model to classify
  the emotional tone of speech.
- **Speaker verification** to determine whether two recordings come from the
  same person using speaker embeddings and cosine similarity.
- **Speech enhancement** to compare acoustic features before and after
  noise removal.

Next up: **Tutorial 2 -- Transcription & Phoneme Analysis**, where we will
transcribe speech to text, perform forced alignment, and analyze phoneme-level
features.